# Notebook Inicial del proyecto

In [0]:
nombre_catalogo = "fintech_finpay"

schemas = [
    "bronze",
    "silver",
    "gold",
    "observability"
]

volume_schema = "bronze"
volume_name = "landing"

grupos = {
    "ingenieria": "ingenieria",
    "riesgo": "riesgo",
    "auditoria": "auditoria"
}

In [0]:
#creamos el catalogo

spark.sql(f"""
CREATE CATALOG IF NOT EXISTS {nombre_catalogo}
""")

DataFrame[]

In [0]:
#creamos los esquemas

for schema in schemas:
    spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS {nombre_catalogo}.{schema}
    """)

In [0]:
#Creamos Volume de landing

spark.sql(f"""
CREATE VOLUME IF NOT EXISTS
{nombre_catalogo}.{volume_schema}.{volume_name}
""")

DataFrame[]

In [0]:
#creamos subdirectorios dentro del volume

subdirs = [
    "users",
    "transactions",
    "payments",
    "logs"
]

for subdir in subdirs:
    dbutils.fs.mkdirs(
        f"/Volumes/{nombre_catalogo}/{volume_schema}/{volume_name}/{subdir}"
    )

# Previamente crear los grupos manualmente, no enconre la forma de hacerlo por codigo en la versión gratis

In [0]:
#asignamos permisos al grupo de ingenieria

spark.sql(f"""
GRANT USE CATALOG ON CATALOG {nombre_catalogo}
TO `ingenieria`
""")

for schema in schemas:
    spark.sql(f"""
    GRANT USE SCHEMA, CREATE TABLE, MODIFY
    ON SCHEMA {nombre_catalogo}.{schema}
    TO `ingenieria`
    """)

In [0]:
spark.sql(f"""
GRANT USE CATALOG ON CATALOG {nombre_catalogo}
TO `riesgo`
""")

for schema in ["silver", "gold"]:

    spark.sql(f"""
    GRANT USE SCHEMA
    ON SCHEMA {nombre_catalogo}.{schema}
    TO `riesgo`
    """)

    spark.sql(f"""
    GRANT SELECT
    ON SCHEMA {nombre_catalogo}.{schema}
    TO `riesgo`
    """)

In [0]:
#Asignamos permisos al grupo de auditoria

spark.sql(f"""
GRANT USE CATALOG ON CATALOG {nombre_catalogo}
TO `auditoria`
""")

for schema in ["gold", "observability"]:
    spark.sql(f"""
    GRANT USE SCHEMA
    ON SCHEMA {nombre_catalogo}.{schema}
    TO `auditoria`
    """)

    spark.sql(f"""
    GRANT SELECT
    ON SCHEMA {nombre_catalogo}.{schema}
    TO `auditoria`
    """)

In [0]:
spark.sql(f"""
CREATE VOLUME IF NOT EXISTS
{nombre_catalogo}.default.vol_landing
""")

DataFrame[]

In [0]:
nombre_schema = "default"
nombre_volume = "vol_landing"

directorios = [
    "transactions",
    "merchants",
    "users"
]

for directorio in directorios:
    
    path = f"/Volumes/{nombre_catalogo}/{nombre_schema}/{nombre_volume}/{directorio}"
    
    dbutils.fs.mkdirs(path)

    print(f"Directorio creado: {path}")

Directorio creado: /Volumes/fintech_finpay/default/vol_landing/transactions
Directorio creado: /Volumes/fintech_finpay/default/vol_landing/merchants
Directorio creado: /Volumes/fintech_finpay/default/vol_landing/users


In [0]:
display(
    spark.sql(f"SHOW SCHEMAS IN {nombre_catalogo}")
)

databaseName
bronze
default
gold
information_schema
observability
silver


In [0]:
display(
    spark.sql("SHOW GROUPS")
)

name,directGroup
admins,null
users,null
ingenieria,null
auditoria,null
riesgo,null


In [0]:
display(
    spark.sql("SHOW CATALOGS")
)

catalog
fintech_finpay
samples
system
workspace


In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS fintech_finpay.silver")
 
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS fintech_finpay.silver.transactions (
        transaction_id   STRING    COMMENT 'PK — identificador único de la transacción',
        user_id          STRING    COMMENT 'Identificador del usuario',
        merchant_id      STRING    COMMENT 'Identificador del comercio',
        channel          STRING    COMMENT 'Canal: web | app | pos | kiosko',
        transaction_type STRING    COMMENT 'Tipo: pago | reversa | retiro | payment',
        amount           DOUBLE    COMMENT 'Monto normalizado positivo',
        currency         STRING    COMMENT 'Código de moneda ISO',
        transaction_date DATE      COMMENT 'Fecha normalizada yyyy-MM-dd',
        status           STRING    COMMENT 'Estado: aprobado | rechazado | pendiente | procesado',
        reference_id     STRING    COMMENT 'ID transacción original si es reversa',
        _source_name     STRING,
        _source_format   STRING,
        _schema_version  INTEGER,
        _ingestion_ts    TIMESTAMP,
        _source_file     STRING,
        _silver_ts       TIMESTAMP COMMENT 'Timestamp de procesamiento silver'
    )
    USING DELTA
    COMMENT 'Transactions validadas — capa Silver FinPay'
    TBLPROPERTIES (
        'delta.autoOptimize.optimizeWrite' = 'true',
        'delta.autoOptimize.autoCompact'   = 'true'
    )
""")

DataFrame[]

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS fintech_finpay.silver.quarantine (
        quarantine_id    STRING    NOT NULL,
        event_ts         TIMESTAMP NOT NULL,
        event_date       DATE      NOT NULL,
        source_name      STRING    NOT NULL,
        target_table     STRING    NOT NULL,
        rejection_reason STRING    NOT NULL,
        rejected_field   STRING,
        raw_record       STRING    NOT NULL,
        pipeline_run_id  STRING
    )
    USING DELTA
    COMMENT 'Registros rechazados por calidad — Silver FinPay'
    TBLPROPERTIES (
        'delta.autoOptimize.optimizeWrite' = 'true',
        'delta.autoOptimize.autoCompact'   = 'true'
    )
""")